# Music-to-Dance Generation via Atomic Movements — Google Colab

二段階パイプライン（atomic planner → dance completion）を Colab の GPU で一通り
動かすノートブックです。環境構築・データセット取得・学習・推論・描画・評価まで
カバーしています。

**実行前に:** *ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ → GPU*
を選んでください（T4 で足ります）。

Colab の VM はセッション終了時に消えます。「2. 結果を Google Drive に保存する」で
チェックポイントと出力を Drive にミラーしておくと、切断しても学習をやり直さずに
済みます。

## 1. 環境構築

どのコードが動くかを決めるのは、このノートブックをどこから開いたかではなく
**下の clone セル**です。ブランチからこのノートブックを開いた場合は、`BRANCH` に
同じブランチ名を入れてください。

すでに `/content/AtomicDance` がある場合も、このセルは指定ブランチに強制的に
切り替えます（ローカルの変更は破棄されます）。

In [ ]:
!nvidia-smi || echo "GPU が見つかりません。ランタイム > ランタイムのタイプを変更 > GPU を選んでください"

In [ ]:
import os

REPO_URL = "https://github.com/yamak493/AtomicDance.git"
REPO_DIR = "/content/AtomicDance"
# Colab 対応が main にマージされたら "main" に変更してください。
BRANCH = "claude/laughing-bardeen-73mdcu"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone --branch "{BRANCH}" "{REPO_URL}" $REPO_DIR
else:
    # 既存のクローンを指定ブランチに合わせ直す（BRANCH を変えて再実行したとき用）
    !git -C $REPO_DIR fetch origin "{BRANCH}"
    !git -C $REPO_DIR checkout -B "{BRANCH}" "origin/{BRANCH}"

%cd $REPO_DIR
!git log --oneline -1

チェックアウトしたのが Colab 対応版かどうかを確認します。ここで失敗する場合は
`BRANCH` の指定が違います。

In [ ]:
from pathlib import Path

REQUIRED = ("compat/rotation_conversions.py", "compat/smpl.py",
            "tests/__init__.py", "requirements-colab.txt")
missing = [name for name in REQUIRED if not Path(name).exists()]
if missing:
    raise RuntimeError(
        "Colab 対応版のファイルが見つかりません: {}\n"
        "上のセルの BRANCH を Colab 対応が入ったブランチに変えて実行し直してください。"
        .format(", ".join(missing))
    )
print("Colab 対応版のチェックアウトを確認しました")

Colab には CUDA 版の PyTorch が最初から入っているので、`requirements-colab.txt` は
その周辺で足りないものだけを入れます。リポジトリの `compat/` パッケージが PyTorch3D
（Colab 用のホイールが無く、ソースビルドに数十分かかる）を純 PyTorch 実装で置き換える
ので、ここでコンパイルされるものはありません。

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA 利用可否:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("デバイス:", torch.cuda.get_device_name(0))

GPU 時間を使う前に、テストを流してこのランタイムでコードが動くことを確認します。

In [ ]:
!python -m unittest discover -s tests -t . -v

## 2. 結果を Google Drive に保存する（推奨）

Drive をマウントして `runs/`（チェックポイント）と `outputs/` をそこに置きます。
切断しても消えません。VM のローカルディスクで済ませたい場合は `USE_DRIVE = False`
にしてください。

In [ ]:
from pathlib import Path

USE_DRIVE = True  # ローカルディスクだけで済ませるなら False

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/AtomicDance")
else:
    WORK_DIR = Path("/content/atomicdance-work")

RUNS_DIR = WORK_DIR / "runs"
OUTPUT_DIR = WORK_DIR / "outputs"
for directory in (RUNS_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("チェックポイント ->", RUNS_DIR)
print("出力             ->", OUTPUT_DIR)

## 3. atomic データセットの取得

処理済みの `atomic_aistpp` パッケージには、フレーム同期済みのモーション、35 次元の
音楽特徴、atomic ラベルが入っています。ラベル `1..100` が動きのカテゴリ、`0` が
トランジションです。

下のセルは、ダウンロード・展開・配置を検証付きで行います。アーカイブの中の
ディレクトリ構成がどうなっていても `data/atomic_aistpp/` に揃えます。

In [ ]:
import shutil
import zipfile
from pathlib import Path

DATASET_URL = "https://drive.google.com/file/d/1ETsaetMMWeKV3_E3Lr40BdybAsUAG8WM/view"
ARCHIVE = Path("/content/atomic_aistpp.zip")
STAGING = Path("/content/atomic_aistpp_extracted")
TARGET = Path("data/atomic_aistpp")

if (TARGET / "train" / "motion.npy").exists():
    print("配置済み:", TARGET.resolve())
else:
    if not ARCHIVE.exists():
        !pip install -q --upgrade gdown
        !gdown --fuzzy "{DATASET_URL}" -O "{ARCHIVE}"

    if not ARCHIVE.exists():
        raise RuntimeError(
            "ダウンロードできませんでした。下の「手動ダウンロード」の手順を使ってください。"
        )
    if not zipfile.is_zipfile(ARCHIVE):
        preview = ARCHIVE.read_bytes()[:300]
        ARCHIVE.unlink()
        raise RuntimeError(
            "ZIP ではないファイルが落ちてきました（Drive のダウンロード上限が原因のことが"
            "多いです）。下の「手動ダウンロード」を使ってください。\n先頭バイト: {!r}"
            .format(preview)
        )

    shutil.rmtree(STAGING, ignore_errors=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        archive.extractall(STAGING)

    # アーカイブ内のどの階層に train/ があっても拾う
    found = sorted(STAGING.rglob("train/motion.npy"))
    if not found:
        listing = sorted(str(path.relative_to(STAGING)) for path in STAGING.rglob("*"))
        raise RuntimeError(
            "展開結果に train/motion.npy がありません。中身:\n  "
            + "\n  ".join(listing[:40])
        )
    source = found[0].parent.parent

    TARGET.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(TARGET, ignore_errors=True)
    shutil.move(str(source), str(TARGET))
    print("配置しました:", TARGET.resolve())

print(sorted(path.name for path in TARGET.iterdir()))

**手動ダウンロード（`gdown` が失敗したとき）**

1. [データセットのリンク](https://drive.google.com/file/d/1ETsaetMMWeKV3_E3Lr40BdybAsUAG8WM/view?usp=sharing)
   をブラウザで開いてダウンロードする
2. Colab 左の**ファイル**ペインに ZIP をドラッグして `/content/atomic_aistpp.zip`
   として置く（あるいは自分の Drive に入れて `/content/drive/MyDrive/...` からコピー）
3. 上のセルを再実行する（ZIP が既にあればダウンロードは飛ばします）

In [ ]:
import json
from pathlib import Path

import numpy as np

root = Path("data/atomic_aistpp")
for split in ("train", "test"):
    motion = np.load(root / split / "motion.npy", mmap_mode="r")
    music = np.load(root / split / "music.npy", mmap_mode="r")
    labels = np.load(root / split / "labels.npy", mmap_mode="r")
    names = json.loads((root / split / "names.json").read_text())
    print(split, "motion", motion.shape, "music", music.shape,
          "labels", labels.shape, "names", len(names))

## 4. 学習

どちらのステージも `--data-root data/atomic_aistpp` を読みます。チェックポイントは
再開可能で、切断後は `--resume <チェックポイント>` で続きから流せます。Drive に
書いておく理由がこれです。

Colab の VM は CPU 2 コアなので、既定の 4 より `--workers 2` のほうが速く回ります。
下のバッチサイズは 16 GB の T4 に収まる値です。メモリ不足が出たら下げてください。

まずスモークテストで、ループ全体が数秒で通ることを確認します。

In [ ]:
!python train_atomic.py \
  --stage planner \
  --data-root data/atomic_aistpp \
  --output-dir /tmp/smoke_planner \
  --device cuda \
  --epochs 1 --max-steps 2 --batch-size 2 --workers 2

### Atomic movement planner

In [ ]:
!python train_atomic.py \
  --stage planner \
  --data-root data/atomic_aistpp \
  --output-dir "{RUNS_DIR}/atomic_planner" \
  --device cuda \
  --epochs 20 \
  --batch-size 16 \
  --workers 2

### Dance completion model

200 エポックは数時間かかり、無料枠の Colab セッションより長くなります。分割して
進めてください。このセルを実行したあと、`RUNS_DIR/atomic_completion` にある最新の
チェックポイント（20 エポックごとに保存されます）を `--resume` に指定して再実行します。

In [ ]:
!python train_atomic.py \
  --stage completion \
  --data-root data/atomic_aistpp \
  --output-dir "{RUNS_DIR}/atomic_completion" \
  --device cuda \
  --epochs 200 \
  --batch-size 8 \
  --workers 2

In [ ]:
from pathlib import Path


def newest_checkpoint(directory):
    checkpoints = sorted(Path(directory).glob("*.pt"), key=lambda path: path.stat().st_mtime)
    if not checkpoints:
        raise FileNotFoundError("{} にチェックポイントがありません".format(directory))
    return checkpoints[-1]


PLANNER_CHECKPOINT = newest_checkpoint(RUNS_DIR / "atomic_planner")
COMPLETION_CHECKPOINT = newest_checkpoint(RUNS_DIR / "atomic_completion")
print("planner   :", PLANNER_CHECKPOINT)
print("completion:", COMPLETION_CHECKPOINT)

## 5. ダンスの生成

`--audio-dir` に WAV の入ったフォルダを指定します。atomic データセットには音声が
含まれないので、AIST++ の WAV を `data/edge_aistpp/wavs` に置く（セクション 7 参照）か、
下のセルで自分の曲をアップロードしてください。

AIST++ の命名（`gWA_sBM_c01_d25_mWA4_ch05.wav`）だとファイル名からテンポを読み取ります。
それ以外の名前ならビートトラッキングにフォールバックするので、どちらでも問題ありません。

In [ ]:
# 任意: 自分の .wav をアップロードして生成に使う
from pathlib import Path

from google.colab import files

CUSTOM_AUDIO_DIR = Path("/content/custom_music")
CUSTOM_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
for filename, content in files.upload().items():
    (CUSTOM_AUDIO_DIR / filename).write_bytes(content)
print(sorted(path.name for path in CUSTOM_AUDIO_DIR.glob("*.wav")))

In [ ]:
AUDIO_DIR = "data/edge_aistpp/wavs"  # 自分の曲を使うなら str(CUSTOM_AUDIO_DIR)
GENERATED_DIR = OUTPUT_DIR / "generated"

!python infer_atomic.py \
  --audio-dir "{AUDIO_DIR}" \
  --output-dir "{GENERATED_DIR}" \
  --planner-checkpoint "{PLANNER_CHECKPOINT}" \
  --completion-checkpoint "{COMPLETION_CHECKPOINT}" \
  --data-root data/atomic_aistpp \
  --device cuda \
  --max-frames 150 \
  --inference-batch-size 4

## 6. 結果を描画する

`skeleton_render` はスケルトンの GIF を書き出し、音声と合わせて MP4 にします
（ffmpeg は Colab に入っています）。描画は CPU 処理なので、試行中はクリップを
短くしておくと快適です。

In [ ]:
import pickle
from pathlib import Path

import numpy as np

from vis import skeleton_render

generated = sorted(Path(GENERATED_DIR).glob("*.pkl"))
print("生成されたモーション:", len(generated))

motion_path = generated[0]
with open(motion_path, "rb") as handle:
    data = pickle.load(handle)

frames = 150  # 30 FPS で 5 秒
render_dir = OUTPUT_DIR / "renders"
skeleton_render(
    np.asarray(data["full_pose"])[:frames],
    epoch="atomic",
    out=str(render_dir),
    name=data["audio_path"],
    sound=True,
    contact=np.asarray(data["contacts"])[:frames],
)
print(sorted(path.name for path in render_dir.glob("*.mp4")))

In [ ]:
import base64
from pathlib import Path

from IPython.display import HTML

video_path = sorted(Path(render_dir).glob("*.mp4"))[0]
encoded = base64.b64encode(video_path.read_bytes()).decode()
HTML('<video width=480 controls><source src="data:video/mp4;base64,{}" type="video/mp4"></video>'.format(encoded))

## 7. 評価（任意）

評価指標は生成モーションを AIST++ の正解と比較するので、このセクションだけは
プロジェクトの配布物に含まれない、それぞれ別ライセンスの素材が 2 つ必要です。

- AIST++ のモーション PKL と WAV を `data/edge_aistpp/{motions,wavs}` に配置
  （[AIST++ 公式サイト](https://google.github.io/aistplusplus_dataset/)）
- `SMPL_MALE.pkl` を `smpl/SMPL_MALE.pkl` に配置
  （[SMPL 公式サイト](https://smpl.is.tue.mpg.de/)）

SMPL の配布ファイルは配列を chumpy オブジェクトとして保存していますが、chumpy は
Colab の Python では import できません。`compat/smpl.py` が chumpy 無しで読むので、
ライセンス取得した `.pkl` をそのまま置けば動きます。

評価器は kinetic / manual 特徴の FID と diversity、そして Beat Alignment Score を
出します。キャッシュを使わず作り直すには `--overwrite-inference --force-extract`
を足してください。

In [ ]:
!python -m eval.evaluate \
  --ground-truth-motions data/edge_aistpp/motions \
  --audio-dir data/edge_aistpp/wavs \
  --sequence-list data/splits/crossmodal_test.txt \
  --plan-source planner \
  --planner-checkpoint "{PLANNER_CHECKPOINT}" \
  --completion-checkpoint "{COMPLETION_CHECKPOINT}" \
  --atomic-data-root data/atomic_aistpp \
  --smpl-model smpl/SMPL_MALE.pkl \
  --device cuda:0 \
  --max-inference-frames 150 \
  --inference-batch-size 4 \
  --workers 2 \
  --inference-output "{OUTPUT_DIR}/eval_generated" \
  --cache-dir "{OUTPUT_DIR}/eval_cache" \
  --output "{OUTPUT_DIR}/results_planner.json"

In [ ]:
import json

print(json.dumps(json.loads((OUTPUT_DIR / "results_planner.json").read_text()), indent=2))

## トラブルシューティング

**`ImportError: Start directory is not importable: '.../tests'`** —
`main` をチェックアウトしています。`main` には `tests/__init__.py` が無く、
Python 3.11 以降の unittest は名前空間パッケージを探索できません。clone セルの
`BRANCH` を Colab 対応ブランチにして実行し直してください。

**`FileNotFoundError: data/atomic_aistpp/train/motion.npy`** —
データセットの配置が済んでいません。セクション 3 のセルを実行してください。
`gdown` が Drive のダウンロード上限に当たった場合は、そこに書いてある手動
ダウンロードの手順を使ってください。

**`ModuleNotFoundError: No module named 'compat'`** —
リポジトリのルートから実行していません。`%cd /content/AtomicDance` を実行してください。

**`CUDA out of memory`** — `--batch-size`（学習）や `--inference-batch-size` を
下げて、ランタイムを再起動してアロケータのキャッシュを解放してください。

**学習中にセッションが切れた** — 環境構築のセルを流し直し、`train_atomic.py` に
`--resume <チェックポイント>` を渡します。チェックポイントを Drive に書いていた
場合のみ可能です。

**MP4 が出力されない** — `skeleton_render` は ffmpeg を呼びますが終了ステータスを
見ていません。PKL に記録された音声パスがこの VM にまだ存在するか確認してください。
WAV がローカルディスクに置かれていて消えた場合は、推論からやり直します。